# Experiment Notebook: Pharmacy Constraint-Aware Delivery Batching

This notebook evaluates the **Constraint-Aware Delivery Batching System** against a **Naïve Baseline Algorithm** across a synthetic dataset of **500 time-sensitive pharmacy orders** and **20 riders**.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Add root project directory to sys.path
root_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(root_dir))

from src.data_generator import load_dataset_from_csv
from src.evaluation import run_comparative_experiment
from src.constraints import validate_batch_constraints
from src.models import Order, Rider, Product

## 1. Load Synthetic Pharmacy Dataset

In [ ]:
products, riders, orders = load_dataset_from_csv()
product_map = {p.product_id: p for p in products}
print(f"Dataset Summary: {len(orders)} orders, {len(riders)} riders, {len(products)} products.")

## 2. Run Comparative Experiment: Baseline vs Constraint-Aware System

In [ ]:
results = run_comparative_experiment(
    orders=orders,
    riders=riders,
    product_map=product_map,
    current_time_min=0.0,
    target_distance_saved_pct=10.0
)

display(results['comparison_table'])
print("\nEvaluation Result:", results['explanation'])

## 3. Visualize Experiment Metrics

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Distance comparison
metrics = ['Naïve Baseline', 'Constraint-Aware System']
distances = [results['baseline_metrics']['total_distance_km'], results['optimized_metrics']['total_distance_km']]
ax1.bar(metrics, distances, color=['#e74c3c', '#2ecc71'])
ax1.set_ylabel('Total Distance (km)')
ax1.set_title('Total Delivery Distance Comparison')
for i, v in enumerate(distances):
    ax1.text(i, v + 20, f"{v:.1f} km", ha='center', fontweight='bold')

# Violations comparison
violations_base = results['baseline_metrics']['late_deliveries'] + results['baseline_metrics']['product_violations'] + results['baseline_metrics']['pickup_violations'] + results['baseline_metrics']['capacity_violations'] + results['baseline_metrics']['workload_violations']
violations_opt = results['optimized_metrics']['late_deliveries'] + results['optimized_metrics']['product_violations'] + results['optimized_metrics']['pickup_violations'] + results['optimized_metrics']['capacity_violations'] + results['optimized_metrics']['workload_violations']

ax2.bar(metrics, [violations_base, violations_opt], color=['#e74c3c', '#2ecc71'])
ax2.set_ylabel('Total Hard Constraint Violations')
ax2.set_title('Constraint Violations Count')
for i, v in enumerate([violations_base, violations_opt]):
    ax2.text(i, v + 0.5, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 4. Test Failure Case Scenarios

Verify that candidate batches violating constraints produce structured rejection diagnostics.

In [ ]:
# Example Case: Incompatible products (ColdChain + Hazmat)
o_cold = Order("O-TEST-1", "P-COLD", "Insulin", "Loc 1", 37.78, -122.41, 0, 0, 120, 1, "HIGH", "ColdChain", "COLD_CHAIN")
o_haz = Order("O-TEST-2", "P-HAZ", "Alcohol", "Loc 2", 37.78, -122.41, 0, 0, 120, 1, "MEDIUM", "Hazmat", "HAZMAT")
test_rider = riders[0]

res = validate_batch_constraints([o_cold, o_haz], test_rider, product_map=product_map)
print(f"Feasible: {res.feasible}")
print(f"Reason: {res.reason}")